# Planeación de la producción en una empresa de aceites

## Propósito

Formular y resolver un problema de programación lineal para determinar el plan semanal de producción de una empresa de aceites comestibles. El ejercicio conserva una complejidad equivalente al ejemplo de planeación de producción: dos productos, tres recursos, costos operativos, capacidad limitada y una restricción de mercado.

Al finalizar, el estudiante podrá identificar variables de decisión, expresiones, función objetivo, restricciones, recursos limitantes y la interpretación económica de una solución óptima.

## 1. Declaración del problema

**Aceites del Sur S.A.S.** produce semanalmente dos mezclas de aceite comestible: aceite **Premium** ($P$) y aceite **Tradicional** ($T$). Ambos productos utilizan aceite vegetal base y pasan por dos áreas especializadas: refinación y envasado.

El aceite Premium se vende a 310 unidades monetarias por lote. Cada lote requiere 12 litros de aceite base, 2 horas de refinación y 1 hora de envasado. Debido al segmento de mercado al que está dirigido, su demanda está limitada a 35 lotes por semana.

El aceite Tradicional se vende a 220 unidades monetarias por lote. Cada lote requiere 9 litros de aceite base, 1 hora de refinación y 1,5 horas de envasado. Para efectos del ejercicio, su demanda se considera ilimitada.

| Producto | Aceite base | Refinación | Envasado | Demanda semanal | Precio por lote |
|:--|--:|--:|--:|--:|--:|
| Premium ($P$) | 12 L | 2 h | 1 h | $\leq 35$ | 310 |
| Tradicional ($T$) | 9 L | 1 h | 1,5 h | Ilimitada | 220 |

El aceite vegetal base cuesta 8 unidades monetarias por litro y puede comprarse sin límite, pero es perecedero una vez abierto. Por ello, la empresa desea pedir solamente la cantidad que utilizará durante la semana. El área de refinación dispone de 90 horas semanales y cuesta 45 unidades monetarias por hora. El área de envasado dispone de 100 horas semanales y cuesta 35 unidades monetarias por hora.

| Recurso | Disponibilidad | Costo |
|:--|--:|--:|
| Aceite vegetal base | Ilimitada | 8/L |
| Refinación | 90 h/semana | 45/h |
| Envasado | 100 h/semana | 35/h |

La empresa quiere maximizar su beneficio bruto semanal.

### Preguntas

1. ¿Cuántos lotes de aceite Premium y Tradicional se deben producir?
2. ¿Cuántos litros de aceite vegetal base se deben comprar?
3. ¿Cuántas horas de refinación y envasado se utilizarán?
4. ¿Cuál será el ingreso, el costo operativo y el beneficio bruto semanal?
5. ¿Qué recursos limitan el plan de producción?

## 2. Variables de decisión

Se definen variables para la producción y para la cantidad utilizada de cada recurso.

| Variable | Descripción | Límite inferior | Límite superior |
|:--:|:--|--:|--:|
| $y_P$ | Lotes de aceite Premium | 0 | 35 |
| $y_T$ | Lotes de aceite Tradicional | 0 | — |
| $x_M$ | Litros de aceite base utilizados | 0 | — |
| $x_R$ | Horas de refinación utilizadas | 0 | 90 |
| $x_E$ | Horas de envasado utilizadas | 0 | 100 |

Se permiten valores continuos porque un lote representa una cantidad estandarizada de producción que puede fraccionarse. Si la empresa solo pudiera fabricar lotes completos, $y_P$ y $y_T$ deberían declararse como variables enteras.

## 3. Función objetivo y expresiones económicas

El ingreso semanal es:
ingreso = 310*Yp+220*Yt



El costo de aceite base, refinación y envasado es:
costo = 8*Xm+45Xr+35Xe


El beneficio bruto corresponde a la diferencia entre ingresos y costos:
beneficio => 310*Yp+220*Yt-8*Xm-45Xr-35Xe



## 4. Restricciones

La producción no puede consumir más recursos de los que se compran o habilitan:

* Aceite base:  12Yp + 9Yt  <= Xm
* refinacion:   2yp  + Yt     <= Xr
* Envasado:     Yp   + 1.5Yt    <= Xe

Las cotas de las variables representan disponibilidad y demanda:

           *     0 <= Xr <=90
           *     0 <= Xe <= 100
           *     0 <= Yp <= 35
           *     Yt >= 0
           *     Xm >= 0


Preparación del entorno

In [49]:
try:
  import pyomo.environ as pyo
  import highspy
except ImportError:
  %pip install -q pyomo highspy
  import pyomo.environ as pyo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt




Implementación del modelo

In [50]:
modelo = pyo.ConcreteModel('Planeacion de produccion de aceites')

# Variables

In [51]:
modelo.y_p = pyo.Var(domain=pyo.NonNegativeReals,bounds=(0,35))
modelo.y_t = pyo.Var(domain=pyo.NonNegativeReals)
modelo.x_m = pyo.Var(domain=pyo.NonNegativeReals)
modelo.x_r = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0,90))
modelo.x_e = pyo.Var(domain=pyo.NonNegativeReals, bounds=(0,100))

Expresiones económicas

In [52]:
modelo.ingreso = pyo.Expression(expr = 310*modelo.y_p + 220*modelo.y_t)
modelo.costo = pyo.Expression(expr = 8*modelo.x_m + 45*modelo.x_r + 35*modelo.x_e)
modelo.beneficio = pyo.Objective(expr = modelo.ingreso - modelo.costo, sense=pyo.maximize)


Restricciones

In [53]:
modelo.aceite_base = pyo.Constraint(expr = 12*modelo.y_p + 9*modelo.y_t <= modelo.x_m)
modelo.refinacion = pyo.Constraint(expr = 2*modelo.y_p + modelo.y_t <= modelo.x_r)
modelo.envasado = pyo.Constraint(expr = modelo.y_p + 1.5*modelo.y_t <= modelo.x_e)

Solucionador

In [54]:
solucionador = pyo.SolverFactory('appsi_highs')
resultados = solucionador.solve(modelo)


In [55]:
print('Lotes Premium:', pyo.value(modelo.y_p))
print('Lotes Tradicional:', pyo.value(modelo.y_t))
print('Lotes Aceite base:', pyo.value(modelo.x_m))
print('Refinacion:', pyo.value(modelo.x_r))
print('Envasado:', pyo.value(modelo.x_e))

print('ingreso:', pyo.value(modelo.ingreso))
print('Costo:', pyo.value(modelo.costo))
print('Beneficio:', pyo.value(modelo.beneficio))


Lotes Premium: 17.5
Lotes Tradicional: 55.0
Lotes Aceite base: 705.0
Refinacion: 90.0
Envasado: 100.0
ingreso: 17525.0
Costo: 13190.0
Beneficio: 4335.0


Holguras

In [56]:
valores = {
'Lotes Premium': pyo.value(modelo.y_p),
'Lotes Tradicional': pyo.value(modelo.y_t),
'Lotes Aceite base': pyo.value(modelo.x_m),
'Refinacion': pyo.value(modelo.x_r),
'Envasado': pyo.value(modelo.x_e),
'ingreso': pyo.value(modelo.ingreso),
'Costo': pyo.value(modelo.costo),
'Beneficio': pyo.value(modelo.beneficio)
}
resultados = pd.DataFrame({'Indicador':valores.keys(), 'valor':valores.values()})
resultados.style.format({'valor':'{:,.2f}'})


,Indicador,valor
0,Lotes Premium,17.50
1,Lotes Tradicional,55.00
2,Lotes Aceite base,705.00
3,Refinacion,90.00
4,Envasado,100.00
5,ingreso,"17,525.00"
6,Costo,"13,190.00"
7,Beneficio,"4,335.00"


In [57]:
holguras = pd.DataFrame({
    'Restricción': ['Refinación', 'Envasado', 'Demanda Premium'],
    'Uso': [
        # 12*modelo.y_p + 9*modelo.y_t <= modelo.x_m
        # 2*modelo.y_p + modelo.y_t <= modelo.x_r)
        # modelo.y_p + 1.5*modelo.y_t <= modelo.x_e

        2*valores['Lotes Premium'] + valores['Lotes Tradicional'],
        valores['Lotes Premium'] + 1.5*valores['Lotes Tradicional'],
        valores['Lotes Premium'],
    ],
    'Disponibilidad': [90, 100, 35],
})
holguras['Holgura'] = holguras['Disponibilidad'] - holguras['Uso']
holguras.style.format({'Uso': '{:.2f}', 'Disponibilidad': '{:.2f}', 'Holgura': '{:.2f}'})

,Restricción,Uso,Disponibilidad,Holgura
0,Refinación,90.00,90.00,0.00
1,Envasado,100.00,100.00,0.00
2,Demanda Premium,17.50,35.00,17.50
